# Lesson 05 · New Categories from Old

Reverse the arrows, pair things up, keep only some of them, or promote the arrows themselves to objects. A small kit of constructions turns the categories we have into the categories we need.

This notebook collects every Python block from the optional labs and exercises of [the lesson page](../lesson-05-new-categories-from-old.html), in page order. Run the cells from top to bottom; later cells use names defined in earlier ones.

Python is optional in this course — the lesson is complete without it. *Generated by `tools/sync_code.py`: edit the lesson page, not this file.*

## §8 — Constructions by hand

**`setup.py`** — carried over from Lessons 1 and 4

In [1]:
from itertools import product

class FiniteCategory:
    """objects, arrows (name -> (dom, cod)), a composition table,
       and a chosen identity arrow for each object."""
    def __init__(self, objects, arrows, comp, ident):
        self.obj, self.ar = set(objects), arrows
        self.comp, self.id = comp, ident

    def dom(self, a): return self.ar[a][0]
    def cod(self, a): return self.ar[a][1]
    def composable(self, g, f): return self.dom(g) == self.cod(f)

    def check(self):
        # 1. each identity is typed X -> X
        for X in self.obj:
            assert self.ar[self.id[X]] == (X, X)
        # 2. every composable pair has a composite, correctly typed
        for g, f in product(self.ar, repeat=2):
            if self.composable(g, f):
                gf = self.comp[(g, f)]
                assert self.ar[gf] == (self.dom(f), self.cod(g))
        # 3. identity law on both sides
        for f in self.ar:
            X, Y = self.ar[f]
            assert self.comp[(self.id[Y], f)] == f
            assert self.comp[(f, self.id[X])] == f
        # 4. associativity
        for h, g, f in product(self.ar, repeat=3):
            if self.composable(h, g) and self.composable(g, f):
                assert self.comp[(h, self.comp[(g, f)])] == \
                       self.comp[(self.comp[(h, g)], f)]
        return True

def arrows_between(C, X, Y):
    "all arrows X -> Y of the finite category C"
    return [a for a, (d, c) in C.ar.items() if d == X and c == Y]

def make_poset(objs, leq):
    "a finite preorder as a thin category; the arrow a -> b is named 'a<=b'"
    ar = {f"{a}<={b}": (a, b) for a in objs for b in objs if leq(a, b)}
    comp = {(g, f): f"{ar[f][0]}<={ar[g][1]}"
            for g in ar for f in ar if ar[f][1] == ar[g][0]}
    return FiniteCategory(objs, ar, comp, {a: f"{a}<={a}" for a in objs})

# the category 2 = ( 0 --u--> 1 )
two_ar   = {'i0':(0,0), 'i1':(1,1), 'u':(0,1)}
two_comp = {('i0','i0'):'i0', ('i1','i1'):'i1',
            ('u','i0'):'u',  ('i1','u'):'u'}
Two = FiniteCategory({0,1}, two_ar, two_comp, {0:'i0', 1:'i1'})

# (Z/3, +) as a one-object category
Z3 = FiniteCategory({'*'}, {n: ('*', '*') for n in range(3)},
                    {(a, b): (a + b) % 3
                     for a in range(3) for b in range(3)}, {'*': 0})

**`constructions.py`** — opposite, product, arrow category, slice

In [2]:
def opposite(C):
    ar = {a: (c, d) for a, (d, c) in C.ar.items()}
    comp = {(f, g): gf for (g, f), gf in C.comp.items()}
    return FiniteCategory(C.obj, ar, comp, C.id)

def product_category(C, D):
    objs = {(X, Y) for X in C.obj for Y in D.obj}
    ar = {(f, g): ((C.dom(f), D.dom(g)), (C.cod(f), D.cod(g)))
          for f in C.ar for g in D.ar}
    comp = {(q, p): (C.comp[(q[0], p[0])], D.comp[(q[1], p[1])])
            for q in ar for p in ar if ar[p][1] == ar[q][0]}
    return FiniteCategory(objs, ar, comp,
                          {(X, Y): (C.id[X], D.id[Y]) for X, Y in objs})

def arrow_category(C):
    "objects: arrows of C;  arrows f -> g: squares (a, b) with g.a == b.f"
    ar = {(f, a, b, g): (f, g)
          for f in C.ar for g in C.ar
          for a in arrows_between(C, C.dom(f), C.dom(g))
          for b in arrows_between(C, C.cod(f), C.cod(g))
          if C.comp[(g, a)] == C.comp[(b, f)]}
    comp = {(q, p): (p[0], C.comp[(q[1], p[1])],
                     C.comp[(q[2], p[2])], q[3])
            for q in ar for p in ar if p[3] == q[0]}
    ident = {f: (f, C.id[C.dom(f)], C.id[C.cod(f)], f) for f in C.ar}
    return FiniteCategory(set(C.ar), ar, comp, ident)

def slice_category(C, A):
    "objects: arrows into A;  arrows f -> g: h with g.h == f"
    objs = [f for f in C.ar if C.cod(f) == A]
    ar = {(f, h, g): (f, g)
          for f in objs for g in objs
          for h in arrows_between(C, C.dom(f), C.dom(g))
          if C.comp[(g, h)] == f}
    comp = {(q, p): (p[0], C.comp[(q[1], p[1])], q[2])
            for q in ar for p in ar if p[2] == q[0]}
    return FiniteCategory(set(objs), ar, comp,
                          {f: (f, C.id[C.dom(f)], f) for f in objs})

def size(C):
    return f"{len(C.obj)} objects, {len(C.ar)} arrows"

Three = make_poset({0, 1, 2}, lambda a, b: a <= b)
Div   = make_poset({1, 2, 3, 6}, lambda a, b: b % a == 0)
tests = [("2 op", opposite(Two)),
         ("2 x 2", product_category(Two, Two)),
         ("2 x 3", product_category(Two, Three)),
         ("arrows of 2", arrow_category(Two)),
         ("arrows of 3", arrow_category(Three)),
         ("Div / 2", slice_category(Div, 2)),
         ("Div / 6", slice_category(Div, 6)),
         ("Z3 / *", slice_category(Z3, '*'))]
for name, K in tests:
    assert K.check()
    print(f"{name:12}", size(K))

2 op         2 objects, 3 arrows
2 x 2        4 objects, 9 arrows
2 x 3        6 objects, 18 arrows
arrows of 2  3 objects, 6 arrows
arrows of 3  6 objects, 20 arrows
Div / 2      2 objects, 3 arrows
Div / 6      4 objects, 9 arrows
Z3 / *       3 objects, 9 arrows


## Exercise 14

**`more_constructions.py`**

In [3]:
Z2 = FiniteCategory({'*'}, {n: ('*', '*') for n in range(2)},
                    {(a, b): (a + b) % 2
                     for a in range(2) for b in range(2)}, {'*': 0})
S = slice_category(Z2, '*')
print("Z2 / * :", size(S), "- every hom-set has",
      {len(arrows_between(S, m, n)) for m in S.obj for n in S.obj}, "arrow")

lhs = opposite(product_category(Two, Three))
rhs = product_category(opposite(Two), opposite(Three))
print("same objects:", lhs.obj == rhs.obj,
      " same typed arrows:", lhs.ar == rhs.ar,
      " same composition:", lhs.comp == rhs.comp)

Z2 / * : 2 objects, 4 arrows - every hom-set has {1} arrow
same objects: True  same typed arrows: True  same composition: True
